In [1]:
!pip install jiwer
!pip install evaluate
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.9 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import random
import logging
import evaluate
import torch
import torchaudio
import pytorch_lightning as pl

from tqdm import tqdm
from pathlib import Path
from datasets import load_dataset
from dataclasses import dataclass
from pytorch_lightning import Trainer
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [3]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [6]:
AUDIO_DIR = Path("/content")

In [5]:
!unzip ytc.zip

Streaming output truncated to the last 5000 lines.
  inflating: ytc/uk/G0yfpbiXgpM-245824-2695.wav  
  inflating: __MACOSX/ytc/uk/._G0yfpbiXgpM-245824-2695.wav  
  inflating: ytc/uk/2nhhRpDKXo8-364512-1576.wav  
  inflating: __MACOSX/ytc/uk/._2nhhRpDKXo8-364512-1576.wav  
  inflating: ytc/uk/YtfG95TcUcM-43368-1000.wav  
  inflating: __MACOSX/ytc/uk/._YtfG95TcUcM-43368-1000.wav  
  inflating: ytc/uk/2nhhRpDKXo8-234248-3519.wav  
  inflating: __MACOSX/ytc/uk/._2nhhRpDKXo8-234248-3519.wav  
  inflating: ytc/uk/eIX04amSZ_M-52543-3392.wav  
  inflating: __MACOSX/ytc/uk/._eIX04amSZ_M-52543-3392.wav  
  inflating: ytc/uk/WJ3Lswb5OXA-161432-1975.wav  
  inflating: __MACOSX/ytc/uk/._WJ3Lswb5OXA-161432-1975.wav  
  inflating: ytc/uk/jC2Q_T89YXI-486896-807.wav  
  inflating: __MACOSX/ytc/uk/._jC2Q_T89YXI-486896-807.wav  
  inflating: ytc/uk/YtfG95TcUcM-365552-1704.wav  
  inflating: __MACOSX/ytc/uk/._YtfG95TcUcM-365552-1704.wav  
  inflating: ytc/uk/iTInWOM55EI-1200-3184.wav  
  inflating: __MACO

In [7]:
dataset = load_dataset(
    "nvidia/Granary",
    "uk",
    split="ast"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

uk/ytc/uk_asr.jsonl:   0%|          | 0.00/2.82M [00:00<?, ?B/s]

uk/ytc/uk_ast-en.jsonl:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

Generating asr split: 0 examples [00:00, ? examples/s]

Generating ast split: 0 examples [00:00, ? examples/s]

In [34]:
def split_data(items):
    items = list(items)
    random.shuffle(items)

    test_size = int(0.1 * len(items))
    val_size = int(0.1 * len(items))

    test = items[:test_size]
    val = items[test_size:test_size + val_size]
    train = items[test_size + val_size:]

    return train, val, test

In [35]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\sа-яіїєґ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [36]:
class GranaryDataset(Dataset):
    def __init__(self, items, processor):
        self.items = [
            x for x in items
            if os.path.exists(AUDIO_DIR / x["audio_filepath"])
        ]
        self.processor = processor

    @staticmethod
    def safe_load_audio(path):
        try:
            audio, sr = torchaudio.load(path)
            if audio is None or audio.numel() == 0:
                return None
            return audio, sr
        except Exception:
            return None

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        audio_data = self.safe_load_audio(AUDIO_DIR / item["audio_filepath"])

        if audio_data is None:
            return self.__getitem__((idx + 1) % len(self.items))

        audio, sr = audio_data
        audio = audio.mean(dim=0)

        if sr != 16000:
            audio = torchaudio.functional.resample(audio, sr, 16000)

        target_text = normalize_text(item["answer"])

        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        labels = self.processor.tokenizer(
            target_text,
            return_tensors="pt"
        ).input_ids

        return {
            "input_features": inputs.input_features[0],
            "labels": labels[0]
        }

In [37]:
@dataclass
class DataCollator:
    processor: any

    def __call__(self, batch):
        input_features = [b["input_features"] for b in batch]
        labels = [b["labels"] for b in batch]

        input_features = torch.nn.utils.rnn.pad_sequence(
            input_features, batch_first=True
        )

        attention_mask = torch.ones_like(input_features[:, :, 0])

        labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )

        return {
            "input_features": input_features,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [38]:
class WhisperLightning(pl.LightningModule):
    def __init__(self, model_name, processor, lr=1e-5):
        super().__init__()
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)

        self.model.generation_config.suppress_tokens = []
        self.model.generation_config.forced_decoder_ids = None

        self.processor = processor
        self.lr = lr

    def training_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("train_loss", out.loss, prog_bar=True, on_step=True, on_epoch=True)
        return out.loss

    def validation_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("val_loss", out.loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

In [40]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny",
    language="uk",
    task="translate",
)

items = dataset

train_items, val_items, test_items = split_data(items)

train_ds = GranaryDataset(train_items, processor)
val_ds = GranaryDataset(val_items, processor)

collator = DataCollator(processor)

train_loader = DataLoader(
    train_ds, batch_size=8, num_workers=4, shuffle=True, collate_fn=collator,
)
val_loader = DataLoader(
    val_ds, batch_size=8, num_workers=4, collate_fn=collator,
)

model = WhisperLightning("openai/whisper-tiny", processor)
model.train()

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    # precision="bf16-mixed",
    enable_progress_bar=True,
    log_every_n_steps=1,
)

trainer.fit(model, train_loader, val_loader)

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ WhisperForConditionalGeneration │ 37.8 M │ train │     0 │
└───┴───────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 37.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 37.8 M                                                                                               
Total estimated model params size (MB): 151                                                                        
Modules in train mode: 126                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [45]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def evaluate_model(model, dataloader, processor):
    preds, refs = [], []

    model.eval()
    for batch in tqdm(dataloader):
        with torch.no_grad():
            pred_ids = model.model.generate(
                batch["input_features"].to(model.device),
                language="uk",
                task="translate",
            )

        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        ref_str = processor.batch_decode(labels, skip_special_tokens=True)

        preds.extend(pred_str)
        refs.extend(ref_str)

    return preds, refs

In [42]:
# 1. для інференсу
model.model.save_pretrained("whisper_full")
model.processor.save_pretrained("whisper_full")

# 2. для resume training
trainer.save_checkpoint("whisper_lightning.ckpt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:pytorch_lightning.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.


In [46]:
next(model.model.parameters()).device

device(type='cuda', index=0)

In [44]:
model.to("cuda")

WhisperLightning(
  (model): WhisperForConditionalGeneration(
    (model): WhisperModel(
      (encoder): WhisperEncoder(
        (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 384)
        (layers): ModuleList(
          (0-3): 4 x WhisperEncoderLayer(
            (self_attn): WhisperAttention(
              (k_proj): Linear(in_features=384, out_features=384, bias=False)
              (v_proj): Linear(in_features=384, out_features=384, bias=True)
              (q_proj): Linear(in_features=384, out_features=384, bias=True)
              (out_proj): Linear(in_features=384, out_features=384, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (fc2): Linea

In [47]:
test_ds = GranaryDataset(test_items, processor)
test_loader = DataLoader(
    test_ds, batch_size=8, num_workers=4, collate_fn=collator,
)

preds, refs = evaluate_model(model, test_loader, processor)

indices = random.sample(range(len(preds)), 10)
print("Sample predictions:")
for idx in indices:
    print(f"Ref: {refs[idx]}")
    print(f"Pred: {preds[idx]}")
    print("-" * 20)

wer = wer_metric.compute(predictions=preds, references=refs)
cer = cer_metric.compute(predictions=preds, references=refs)
print(f"WER: {wer}, CER: {cer}")

100%|██████████| 36/36 [01:08<00:00,  1.90s/it]

Sample predictions:
Ref: no just like fernhow we wont write either and well there are proper styles and there are improper ones the dead the outdated or something else we dont want to be like that thats the main thing he said
Pred: no just like the warhead we will not write about it and you are right stylists and you are not postsoviet stylists there is no way we will have a project like this so they say to say
--------------------
Ref: he has a fairly close time now for the translation of some resonant book in the west and we have a fairly short one of course we are not so interested in us that our books are translated so quickly but some important texts some important for example monographs or some popular science things are translated we already notice fairly quickly artistic works
Pred: the number of the artists now is now time for the example of which the original book is in the corner and we are quite short of the fact that our artists were also used to it but some of the importa

In [ ]:
preds

In [ ]:
refs